This notebook is about boosting algorithm and adaboost.
Adaboost is a boosting algorithm that combines multiple weak learners to create a strong learner. 
It works by assigning weights to each training instance and iteratively adjusting these weights based on the performance of the weak learners.
The final model is a weighted sum of the weak learners, where the weights are determined by their performance on the training data. Adaboost can be used for both classification and regression tasks, and it is particularly effective in handling imbalanced datasets.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as smns

In [7]:
df=pd.read_csv("diabetes.csv")
df.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [5]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [8]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [9]:
df['Insulin'].value_counts()

Insulin
0      374
105     11
130      9
140      9
120      8
      ... 
178      1
127      1
510      1
16       1
112      1
Name: count, Length: 186, dtype: int64

In [10]:
df['BloodPressure'].value_counts()

BloodPressure
70     57
74     52
78     45
68     45
72     44
64     43
80     40
76     39
60     37
0      35
62     34
66     30
82     30
88     25
84     23
90     22
86     21
58     21
50     13
56     12
54     11
52     11
92      8
75      8
65      7
85      6
94      6
48      5
44      4
96      4
110     3
106     3
100     3
98      3
30      2
46      2
55      2
104     2
108     2
40      1
122     1
95      1
102     1
61      1
24      1
38      1
114     1
Name: count, dtype: int64

In [11]:
columns_to_check=['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
for column in columns_to_check:
    zero_count=(df[column]==0).sum()
    print(f"{column} columns has {zero_count} zero values")

Glucose columns has 5 zero values
BloodPressure columns has 35 zero values
SkinThickness columns has 227 zero values
Insulin columns has 374 zero values
BMI columns has 11 zero values


In [12]:
X=df.drop('Outcome',axis=1)
y=df['Outcome']
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=34)

In [13]:
columns_to_fill=['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
medians={}
for column in columns_to_fill:
    median_value=X_train[X_train[column] != 0][column].median()
    medians[column]=median_value
    X_train[column]=X_train[column].replace(0,median_value)

for column in columns_to_fill:
    X_test[column]=X_test[column].replace(0,medians[column])

In [14]:
X_train.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,4.019544,120.874593,72.402280,29.114007,140.644951,32.408958,0.468114,33.664495
std,3.396996,30.409450,12.038928,8.655074,84.253333,6.775014,0.317674,11.909726
min,0.000000,44.000000,30.000000,7.000000,15.000000,18.200000,0.078000,21.000000
25%,1.000000,99.000000,64.000000,25.000000,125.000000,27.600000,0.241500,24.000000
50%,3.000000,116.500000,72.000000,29.000000,125.000000,32.250000,0.377000,29.500000
75%,6.000000,140.000000,80.000000,32.000000,125.750000,36.500000,0.618000,41.000000
max,15.000000,199.000000,122.000000,99.000000,744.000000,67.100000,2.329000,81.000000


In [15]:
X_test.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
count,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000
mean,3.149351,124.756494,72.324675,29.084416,140.779221,32.636039,0.486877,31.551948
std,3.174207,30.456668,12.363706,9.343798,94.693192,7.281119,0.381730,11.019912
min,0.000000,57.000000,24.000000,10.000000,14.000000,19.100000,0.084000,21.000000
25%,1.000000,103.000000,65.250000,25.000000,115.000000,27.425000,0.246250,24.000000
50%,2.000000,119.000000,72.000000,29.000000,125.000000,32.275000,0.353500,28.000000
75%,5.000000,140.500000,80.000000,33.000000,138.750000,37.375000,0.647000,37.000000
max,17.000000,198.000000,104.000000,63.000000,846.000000,59.400000,2.420000,67.000000


In [18]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [19]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

model=AdaBoostClassifier()
model.fit(X_train_scaled,y_train)
y_pred=model.predict(X_test_scaled)
accuracy=accuracy_score(y_test,y_pred)
print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(classification_report(y_test,y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred))

Accuracy: 0.8181818181818182
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87       105
           1       0.73      0.67      0.70        49

    accuracy                           0.82       154
   macro avg       0.79      0.78      0.79       154
weighted avg       0.82      0.82      0.82       154

Confusion Matrix:
[[93 12]
 [16 33]]


In [23]:
#hyperparameter tuning
from sklearn.model_selection import GridSearchCV
param_grid={
    'n_estimators':[50,70,100,120,150,200],
    'learning_rate':[0.001,0.01,0.1,1,10]
}
grid_search=GridSearchCV(estimator=AdaBoostClassifier(),param_grid=param_grid,cv=5,verbose=1,n_jobs=-1)
grid_search.fit(X_train_scaled,y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",AdaBoostClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.001, 0.01, ...], 'n_estimators': [50, 70, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 :

In [25]:
grid_search.best_params_

{'learning_rate': 0.1, 'n_estimators': 50}

In [40]:
ada=AdaBoostClassifier(n_estimators=50,learning_rate=1)
ada.fit(X_train_scaled,y_train)
y_pred_ada=ada.predict(X_test_scaled)
accuracy_ada=accuracy_score(y_test,y_pred_ada)
print(f"Accuracy with Best Parameters: {accuracy_ada}")
print("Classification Report with Best Parameters:")
print(classification_report(y_test,y_pred_ada))
print("Confusion Matrix with Best Parameters:")
print(confusion_matrix(y_test,y_pred_ada))

Accuracy with Best Parameters: 0.8181818181818182
Classification Report with Best Parameters:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87       105
           1       0.73      0.67      0.70        49

    accuracy                           0.82       154
   macro avg       0.79      0.78      0.79       154
weighted avg       0.82      0.82      0.82       154

Confusion Matrix with Best Parameters:
[[93 12]
 [16 33]]


In [41]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error

In [42]:
def calculate_model_metrics(true, predicted):
    mse = mean_squared_error(true, predicted)
    r2 = r2_score(true, predicted)
    mae = mean_absolute_error(true, predicted)
    rmse= np.sqrt(mse)
    return mse, r2, mae, rmse

In [43]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "KNeighborsRegressor": KNeighborsRegressor(),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "RandomForestRegressor": RandomForestRegressor()
}

In [44]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train_scaled, y_train)

    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    mse_test, r2_test, mae_test, rmse_test = calculate_model_metrics(y_test, y_test_pred)
    mse_train, r2_train, mae_train, rmse_train = calculate_model_metrics(y_train, y_train_pred)

    print(f"Model: {list(models.values())[i]}")
    print(f"Test {list(models.keys())[i]}: R2={r2_test}, MSE={mse_test}, MAE={mae_test}, RMSE={rmse_test}")
    print(f"Train {list(models.keys())[i]}: R2={r2_train}, MSE={mse_train}, MAE={mae_train}, RMSE={rmse_train}")
    print("-" * 50)
    print("\n")

Model: LinearRegression()
Test LinearRegression: R2=0.3440652248023024, MSE=0.14229989957801295, MAE=0.31333176027826887, RMSE=0.3772265891715654
Train LinearRegression: R2=0.3102772404441215, MSE=0.15826286569454653, MAE=0.3292796982072859, RMSE=0.3978226560850281
--------------------------------------------------


Model: Ridge()
Test Ridge: R2=0.34420715430473225, MSE=0.14226910908678334, MAE=0.3134196987890797, RMSE=0.3771857752975095
Train Ridge: R2=0.3102763039140183, MSE=0.15826308058949654, MAE=0.3293575144721324, RMSE=0.3978229261738148
--------------------------------------------------


Model: Lasso()
Test Lasso: R2=-0.006830942755992053, MSE=0.21842406815987436, MAE=0.44788273615635177, RMSE=0.46735860766639825
Train Lasso: R2=0.0, MSE=0.22945866799647738, MAE=0.45891733599295476, RMSE=0.4790184422300225
--------------------------------------------------


Model: KNeighborsRegressor()
Test KNeighborsRegressor: R2=0.20979591836734712, MSE=0.17142857142857143, MAE=0.301298701

In [ ]:
from lazypredict.Supervised import LazyClassifier
clf = LazyClassifier(verbose=0, ignore_warnings=True)
models, predictions = clf.fit(X_train, X_test, y_train, y_test)
print(models) 

                               Accuracy  Balanced Accuracy   ROC AUC  \
Model                                                                  
AdaBoostClassifier             0.818182           0.779592  0.863848   
QuadraticDiscriminantAnalysis  0.798701           0.765306  0.850146   
GaussianNB                     0.798701           0.754422  0.853450   
SVC                            0.811688           0.753061  0.874052   
ExtraTreesClassifier           0.798701           0.748980  0.855394   
LinearSVC                      0.811688           0.747619  0.870165   
LogisticRegression             0.811688           0.747619  0.869193   
RandomForestClassifier         0.792208           0.744218  0.865792   
LinearDiscriminantAnalysis     0.805195           0.742857  0.870943   
RidgeClassifierCV              0.805195           0.737415  0.871526   
CalibratedClassifierCV         0.805195           0.737415  0.871137   
NearestCentroid                0.753247           0.737415  0.85